# 1. 手法（ベースライン・提案手法）入力のproduce_counterfactual試作

In [ ]:


def produce_counterfactual(factual_batch: Dict, scm: Union[nn.Module, CausalPipeline], do_parent: str, intervention_source: Dataset, ):
    
    factual_batch = {k: v.to(device) for k, v in factual_batch.items()}

    #update with the counterfactual parent
    if force_change:
        possible_values = possible_values[do_parent]
        values = factual_batch[do_parent].cpu()
        if do_parent not in ["digit", "apoE", "slice"]:
            interventions = {do_parent: torch.cat([torch.tensor(np.random.choice(possible_values[different_value(possible_values, value, bins, do_parent)])).unsqueeze(0)
                                                for value in values]).view(-1).unsqueeze(1).to(device)}
        else:
            interventions = {do_parent: torch.cat([torch.tensor(rng.choice(possible_values[torch.where((different_value(possible_values, value, bins, do_parent)).any(dim=1))], axis=0)).unsqueeze(0)
                                                for value in values]).to(device)}
    else:
        batch_size, _ , _ , _ = factual_batch["image"].shape
        idxs = torch.randperm(len(intervention_source))[:batch_size] # select random indices from train set to perform interventions

        interventions = {do_parent: torch.cat([intervention_source[id][do_parent] for id in idxs]).view(-1).unsqueeze(1).to(device)
                        if do_parent not in ["digit", "apoE", "slice"] else torch.cat([intervention_source[id][do_parent].unsqueeze(0).to(device) for id in idxs])}
        
    ### 提案手法は現行の実装でいくパターン ###
    # if isinstance(scm, nn.Module):
    #     abducted_noise = scm.encode(**factual_batch)
    #     counterfactual_batch = scm.decode(interventions, **abducted_noise)
    # elif isinstance(scm, CausalPipeline):
    #     counterfactual_batch = scm.produce_counterfactuals(factual_batch, do_parent, ) # 自分の実装ではscm内部でinterventionsを作成しているため不整合
        
    ### ベースライン実装に近い形にするパターン
    if isinstance(scm, nn.Module):
        abducted_noise = scm.encode(**factual_batch)
        counterfactual_batch = scm.decode(interventions, **abducted_noise)
    elif isinstance(scm, CausalPipeline):
        diffused_noise = scm.diffuse(factual_batch)
        counterfactual_batch = scm.denoise(interventions, diffused_noise)
        
    return counterfactual_batch
        


In [6]:
import torch

intervention = torch.randn(100, 1)
z = torch.randn(100, 4)
z[:, 0] = intervention.squeeze(-1)
z.shape

torch.Size([100, 4])

In [14]:
attrs = ['thickness', 'intensity', 'slant', 'width']
dict = {'intensity': torch.randn(100, 1)}
dict = {attrs.index(attr): dict[attr].squeeze(-1) for attr, value in dict.items()}
list(dict.values())[0].shape

torch.Size([100])

In [2]:
import torch

tensor = torch.randn(100, 1)
tensor.clamp(min=-1.0, max=1.0)
tensor

/home/hashikami/projects/Diff/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[ 2.3955e+00],
        [-1.0465e+00],
        [-4.2841e-01],
        [-5.6697e-01],
        [-3.6048e-01],
        [ 6.3128e-01],
        [ 7.7659e-01],
        [-4.5453e-01],
        [ 3.5109e-01],
        [-5.8126e-01],
        [ 2.4941e-01],
        [-4.1842e-01],
        [ 3.8266e-01],
        [-8.3135e-01],
        [ 9.4435e-01],
        [-1.7830e+00],
        [-9.7032e-01],
        [ 5.9293e-02],
        [-1.9931e-01],
        [ 1.2178e+00],
        [ 1.6141e+00],
        [ 1.9570e+00],
        [ 2.0004e+00],
        [ 3.8297e-01],
        [ 5.0206e-01],
        [ 1.7835e+00],
        [ 1.1666e+00],
        [-6.0747e-01],
        [ 1.0988e+00],
        [ 1.1078e+00],
        [-5.7645e-01],
        [-7.6237e-01],
        [ 9.4589e-02],
        [ 8.0875e-01],
        [ 1.3500e+00],
        [ 1.6021e+00],
        [-1.2839e+00],
        [-7.5037e-01],
        [-5.3049e-01],
        [-1.2799e+00],
        [-1.6078e+00],
        [-2.4850e-01],
        [-6.4242e-01],
        [-2

In [4]:
# B_trueの読み込み

import os
import numpy as np
import igraph as ig

os.chdir("/home/hashikami/projects/Diff/")

B_true_path = "dataset/B_true.npy"
B_true = np.load(B_true_path)
G = ig.Graph.Adjacency(B_true.tolist(), mode='directed')
ordered_vertices = G.topological_sorting()
ordered_vertices
        

[0, 1, 2, 3]

In [26]:
B_true

array([[0, 1, 1, 1],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]])

In [19]:
# shape=(B, dim)次元のnumpy arrayの各dimごとにその値のスケールを[-1, 1]に正規化する

import numpy as np

def normalize(array: np.ndarray) -> np.ndarray:
    
    arrays = []
    for i in range(array.shape[1]):
        min_val = np.min(array[:, i])
        max_val = np.max(array[:, i])
        temp = (array[:, i] - min_val) / (max_val - min_val)
        scaled_array = 2 * temp - 1
        arrays.append(scaled_array)
    return np.array(arrays).T


In [20]:
# スケールの異なるshape=(B, 1)次元のarrayをdim個用意してconcat

import numpy as np

array1 = np.random.randn(100, 1)
array2 = np.random.randn(100, 1) * 0.1
array3 = np.random.randn(100, 1) * 0.01

array = np.concatenate([array1, array2, array3], axis=1)
normalized_array = normalize(array)
normalized_array


array([[-0.12495981,  0.23029036, -0.39795033],
       [-0.32617529,  0.75596948, -0.38455056],
       [-0.33498992,  0.29047615,  0.23706147],
       [-0.03472773, -0.35347966, -0.34469701],
       [ 0.00233531, -0.21794807, -0.23084079],
       [-1.        , -0.50880623, -0.79761546],
       [ 0.65450398, -0.10721915,  1.        ],
       [ 0.31161391, -0.01688749, -0.35703509],
       [-0.14841636, -0.02809808, -0.1393108 ],
       [-0.71716855,  0.16224733, -0.16485605],
       [ 0.59640213, -0.50275864, -0.2572079 ],
       [-0.72083254,  0.17495675,  0.119274  ],
       [ 0.48017268, -0.95463684, -0.00190295],
       [ 0.43350444,  0.09331598,  0.11384441],
       [-0.0794465 , -0.32365696, -0.0690331 ],
       [-0.75715245,  0.50384831, -0.41621824],
       [ 0.07693661, -0.04086497, -0.29346335],
       [-0.0967408 , -0.67785521, -0.64466024],
       [-0.037471  ,  0.24144557, -0.28251513],
       [ 0.13160603,  0.32971639, -0.20995547],
       [ 0.53656192,  0.94891431, -0.455

In [25]:
from dataset.morphomnist import load_morphomnist_like

data_dir = "/home/hashikami/datadrive/morphomnist_all_model/"
_, _, metrics_df = load_morphomnist_like(data_dir)
metrics = metrics_df.to_numpy().astype(np.float32)

normalized_metrics = normalize(metrics)

for i in range(normalized_metrics.shape[1]):
    print(normalized_metrics[:, i].min(), normalized_metrics[:, i].max())




-1.0 1.0
-1.0 1.0
-1.0 1.0
-1.0 1.0


In [32]:
dict = {'a': 1, 'b': 2, 'c': 3}
dict_keys = list(dict.keys())

dict_keys.index('b')

1